In [1]:
!pip install torch_geometric torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.9.0+cu126.html

Looking in links: https://data.pyg.org/whl/torch-2.9.0+cu126.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 104.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 121.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 71.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.2 MB/s eta 0:00:00


In [2]:
import torch
import optuna
import torch.nn as nn
import torch.nn.functional as F
import json

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
from sklearn import metrics

In [3]:
class GIN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, num_classes=1):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()

        for i in range(num_layers):
            in_dim = input_dim if i == 0 else hidden_dim
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))

        self.lin1 = nn.Linear(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        h_graph = 0
        for conv, batch_norm in zip(self.convs, self.batch_norms):
            x = F.relu(batch_norm(conv(x, edge_index)))
            x = F.dropout(x, self.dropout, training=self.training)
            h_graph = h_graph + global_add_pool(x, batch)  # sum over layers

        h = F.relu(self.lin1(h_graph))
        h = F.dropout(h, self.dropout, training=self.training)
        return self.classifier(h).view(-1)

In [4]:
train_dataset = torch.load('/kaggle/input/datasets/samuelvarchol/graphs-with-autgrpg-1/train_dataset_7_features.pt',weights_only=False)
val_dataset = torch.load('/kaggle/input/datasets/samuelvarchol/graphs-with-autgrpg-1/val_dataset_7_features.pt',weights_only=False)

number_of_features = train_dataset[0].num_node_features

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
def objective(trial: optuna.Trial):
    global model, optimizer, criterion, scheduler, train_loader, val_loader

    hidden_dim = trial.suggest_categorical('hidden_dim', [64, 128, 256, 512])
    num_layers = trial.suggest_int('num_layers', 2, 6)
    dropout = trial.suggest_float('dropout', 0.0, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    model = GIN(number_of_features, hidden_dim=hidden_dim, num_layers=num_layers, dropout=dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
    )

    n_epochs = 50
    best_val_acc = 0.0
    patience = 10
    patience_counter = 0

    trial_history = []
    
    for epoch in range(1, n_epochs + 1):
        train_loss = train()
        train_acc, train_f1 = test(train_loader)
        val_acc, val_f1 = test(val_loader)
        
        scheduler.step(val_acc)

        trial_history.append({
            'trial': trial.number,
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'lr': optimizer.param_groups[0]['lr']
        })
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
        else:
            patience_counter += 1
        
        trial.report(val_acc, epoch)
        
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        if patience_counter >= patience:
            print(f"Trial {trial.number}: Early stopping at epoch {epoch}")
            break
        
        if epoch % 10 == 0:
            print(f"Trial {trial.number} | Epoch {epoch:02d} | "
                  f"Train Loss: {train_loss:.4f} | "
                  f"Val Acc: {val_acc:.4f}")
    
    return best_val_acc

In [6]:
def train():
    model.train()
    total_loss = 0

    for data in train_loader:
        data = data.to(device)
        out = model(data)
        loss = criterion(out, data.y.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.num_graphs
        
    return total_loss / len(train_loader.dataset)


@torch.no_grad()
def test(loader):
    model.eval()
    predictions = []
    labels = []

    for data in loader:
        data = data.to(device)
        out = model(data)
        pred = (out > 0).float()
        predictions.append(pred.cpu())
        labels.append(data.y.cpu())

    accuracy = metrics.accuracy_score(torch.cat(labels), torch.cat(predictions))
    f1 = metrics.f1_score(torch.cat(labels), torch.cat(predictions))

    return accuracy, f1

In [7]:
study = optuna.create_study(
    direction="maximize", 
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
    study_name="GIN for partial automorphism extension problem with 7 features dataset")

study.optimize(objective, n_trials=100)

trials_df = study.trials_dataframe()
trials_df.to_csv("/kaggle/working/optuna_trials_summary.csv", index=False)

trial = study.best_trial
print(f"  Value (Val Acc): {trial.value:.4f}")
print("\n  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

best_params = study.best_params
config_path = "/kaggle/working/best_config.json"
with open(config_path, "w") as f:
    json.dump(best_params, f, indent=4)

[I 2026-03-15 00:10:29,939] A new study created in memory with name: GIN for partial automorphism extension problem with 7 features dataset


Trial 0 | Epoch 10 | Train Loss: 0.5895 | Val Acc: 0.6872
Trial 0 | Epoch 20 | Train Loss: 0.5545 | Val Acc: 0.7204
Trial 0 | Epoch 30 | Train Loss: 0.5114 | Val Acc: 0.7423
Trial 0 | Epoch 40 | Train Loss: 0.4843 | Val Acc: 0.7489


[I 2026-03-15 00:16:38,718] Trial 0 finished with value: 0.7536421672941561 and parameters: {'hidden_dim': 256, 'num_layers': 5, 'dropout': 0.3200779744073462, 'lr': 0.0003799473712257059, 'weight_decay': 1.2888494059014622e-06, 'batch_size': 128}. Best is trial 0 with value: 0.7536421672941561.


Trial 0 | Epoch 50 | Train Loss: 0.4686 | Val Acc: 0.7536
Trial 1 | Epoch 10 | Train Loss: 0.5731 | Val Acc: 0.7067
Trial 1 | Epoch 20 | Train Loss: 0.5246 | Val Acc: 0.7351
Trial 1 | Epoch 30 | Train Loss: 0.4807 | Val Acc: 0.7441
Trial 1 | Epoch 40 | Train Loss: 0.4521 | Val Acc: 0.7520


[I 2026-03-15 00:21:26,779] Trial 1 finished with value: 0.7727942380094942 and parameters: {'hidden_dim': 512, 'num_layers': 2, 'dropout': 0.1267754037752089, 'lr': 0.000747844026348116, 'weight_decay': 2.2043256969420385e-06, 'batch_size': 128}. Best is trial 1 with value: 0.7727942380094942.


...

Trial 97 | Epoch 10 | Train Loss: 0.5691 | Val Acc: 0.6885


[I 2026-03-15 06:04:14,081] Trial 97 pruned. 
[I 2026-03-15 06:05:49,198] Trial 98 pruned. 


Trial 99 | Epoch 10 | Train Loss: 0.5663 | Val Acc: 0.7058


[I 2026-03-15 06:07:17,708] Trial 99 pruned. 


  Value (Val Acc): 0.7818

  Params: 
    hidden_dim: 128
    num_layers: 6
    dropout: 0.01619075386111171
    lr: 0.0009966845068403428
    weight_decay: 1.0793701156948604e-06
    batch_size: 64
